# 3B Stage 2: tie-aware DPO-D with Stage 1 retention

This run starts from the existing **3B Stage 1 best checkpoint (step 32)** and trains only Stage 2.

- 538 boundary/overlap pairs are clear wins.
- 292 zero-literal-token-overlap C3/C2 pairs are Davidson ties.
- Stage 1 chosen completions provide a training-only retention anchor; global held-out questions are excluded.
- Checkpoints are selected by `newly_correct - 2 * newly_wrong`, subject to at least 98% retention.
- The output uses a new directory and does not modify the previous Stage 2 runs.


In [ ]:
from pathlib import Path
import importlib
import json
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT = Path("/home/dinh-tuan/Documents/Project/BioASQ_MRES/Task-Structured Counterfactual Preference Mining")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import cse_dpo.train_factoid_three_stage_dpo as trainer
trainer = importlib.reload(trainer)

Config = trainer.Config
load_stage_rows = trainer.load_stage_rows
run_three_stage = trainer.run_three_stage

STAGE1_BEST = PROJECT_ROOT / "Artifacts/cse_dpo/three_stage_dpo_qwen25_3b_c3_c1_curriculum_semantic_full_bp2048/stage_1_concept_learning/checkpoints/step_32"
TIE_DATA_ROOT = PROJECT_ROOT / "Artifacts/cse_dpo/class_judgments/candidate_banks_train_first400_c3_c1_v1_gold_c3/staged_curriculum_pairs_dpo_d_ties_weak_surface_v1"
OUTPUT_ROOT = PROJECT_ROOT / "Artifacts/cse_dpo/three_stage_dpo_qwen25_3b_stage2_dpo_d_retention_from_stage1_step32_bp2048"

CFG = Config(
    model_preset="qwen25_3b",
    dataset_preset="legacy_first400_dpo_d_ties",
    staged_root=str(TIE_DATA_ROOT),
    initial_adapter=str(STAGE1_BEST),
    output_root=str(OUTPUT_ROOT),
    smoke_test=False,
    skip_stages=["concept_learning"],
    stop_after_stage="format_alignment",
    evaluate_generated_dev=True,
    generated_eval_max_seq_length=4096,
    generated_eval_max_new_tokens=64,
    stage_settings={
        "format_alignment": {
            "objective": "dpo_d",
            "dpo_d_nu": 1.0,
            "tie_label_field": "preference_label",
            "beta": 0.05,
            "learning_rate": 2e-7,
            "epochs": 4,
            "batch_size": 1,
            "gradient_accumulation": 8,
            "max_length": 4096,
            "train_backprop_max_length": 2048,
            "optimizer": "AdamW",
            "weight_decay": 0.0,
            "warmup_fraction": 0.05,
            "max_grad_norm": 1.0,
            "lora_dropout": 0.0,
            "retention_anchor_weight": 1.0,
            "retention_anchor_max_examples": 128,
            "l2sp_weight": 0.1,
            "eval_every_updates": 10,
            "early_stopping_patience": 8,
            "selection_metric": "retention_score",
            "retention_loss_penalty": 2.0,
            "retention_min_rate": 0.98,
        }
    },
    max_length=4096,
    train_backprop_max_length=2048,
)

print(json.dumps({
    "stage1_best": str(STAGE1_BEST),
    "tie_data_root": str(TIE_DATA_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "objective": CFG.stage_settings["format_alignment"]["objective"],
    "selection_metric": CFG.stage_settings["format_alignment"]["selection_metric"],
}, indent=2))


In [ ]:
assert STAGE1_BEST.exists(), STAGE1_BEST
assert TIE_DATA_ROOT.exists(), TIE_DATA_ROOT

stages = load_stage_rows(CFG)
stage2 = stages["format_alignment"]
counts = {}
for row in stage2:
    label = row["preference_label"]
    counts[label] = counts.get(label, 0) + 1

print(json.dumps({
    "output_root_already_exists": OUTPUT_ROOT.exists(),
    "stage1_anchor_source_pairs": len(stages["concept_learning"]),
    "stage2_pairs_before_token_budget": len(stage2),
    "stage2_questions": len({row["question_id"] for row in stage2}),
    "stage2_labels": counts,
    "note": "The trainer will report pairs dropped by the 2048-token backward budget in data_summary.json.",
}, indent=2))


In [ ]:
# This starts training. It evaluates step 0 first, then every 10 optimizer updates.
results = run_three_stage(CFG)
print(json.dumps(results, indent=2, default=str))
